# Vectorization and Normalization

This tutorial demonstrates MASA's observation, vectorization, and normalization wrappers without training a policy. The goal is to see what each wrapper changes in the environment interface.

The matching static docs page is at `docs/Tutorials/Wrappers/Vectorization and Normalization.md`.

## CPU-first setup

Keep the notebook portable and quiet before importing MASA/JAX modules.

In [ ]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

## Imports and factories

We will use a small discrete gridworld, a continuous Cartpole task, and a dict-observation LTL safety environment.

In [ ]:
import numpy as np

from masa.common.ltl import Atom, DFA
from masa.common.utils import make_env
from masa.common.wrappers import (
    DummyVecWrapper,
    FlattenDictObsWrapper,
    NormWrapper,
    OneHotObsWrapper,
    VecNormWrapper,
    VecWrapper,
)
from masa.envs.continuous.cartpole import cost_fn as cartpole_cost_fn
from masa.envs.continuous.cartpole import label_fn as cartpole_label_fn
from masa.envs.tabular.colour_bomb_grid_world import label_fn as colour_bomb_label_fn
from masa.envs.tabular.colour_grid_world import cost_fn as colour_grid_cost_fn
from masa.envs.tabular.colour_grid_world import label_fn as colour_grid_label_fn


def make_colour_grid_env(max_episode_steps=5):
    return make_env(
        "colour_grid_world",
        "cmdp",
        max_episode_steps,
        label_fn=colour_grid_label_fn,
        cost_fn=colour_grid_cost_fn,
        budget=0.0,
    )


def make_cartpole_env(max_episode_steps=5):
    return make_env(
        "cont_cartpole",
        "pctl",
        max_episode_steps,
        label_fn=cartpole_label_fn,
        cost_fn=cartpole_cost_fn,
        alpha=0.01,
    )


def make_never_bomb_dfa():
    dfa = DFA([0, 1], 0, [1])
    dfa.add_edge(0, 1, Atom("bomb"))
    return dfa


def make_colour_bomb_ltl_dict_env(max_episode_steps=5):
    return make_env(
        "colour_bomb_grid_world",
        "ltl_safety",
        max_episode_steps,
        label_fn=colour_bomb_label_fn,
        dfa=make_never_bomb_dfa(),
        obs_type="dict",
    )

## OneHotObsWrapper

Discrete observations are convenient for tabular algorithms. Neural policies usually want vector observations, so `OneHotObsWrapper` converts `Discrete(n)` observations to `Box(n,)` one-hot vectors.

In [ ]:
raw_env = make_colour_grid_env()
print("raw observation_space:", raw_env.observation_space)
raw_env.close()

onehot_env = OneHotObsWrapper(make_colour_grid_env())
obs, info = onehot_env.reset(seed=0)
print("wrapped observation_space:", onehot_env.observation_space)
print("obs shape:", obs.shape)
print("one-hot sum:", float(obs.sum()))
print("info keys:", sorted(info.keys()))

assert onehot_env.observation_space.shape == (81,)
assert obs.shape == (81,)
assert obs.sum() == 1.0
onehot_env.close()

## DummyVecWrapper

`DummyVecWrapper` keeps one environment but presents vector-style lists. This is useful for algorithm code that always expects batches.

In [ ]:
dummy_vec_env = DummyVecWrapper(OneHotObsWrapper(make_colour_grid_env()))
obs_list, info_list = dummy_vec_env.reset(seed=0)
print("n_envs:", dummy_vec_env.n_envs)
print("reset obs list length:", len(obs_list))
print("first obs shape:", obs_list[0].shape)

next_obs, rewards, terminated, truncated, infos = dummy_vec_env.step(0)
print("step rewards:", rewards)
print("step terminated:", terminated)
print("step truncated:", truncated)

assert dummy_vec_env.n_envs == 1
assert len(obs_list) == len(info_list) == 1
assert len(next_obs) == len(rewards) == len(terminated) == len(truncated) == len(infos) == 1
dummy_vec_env.close()

## VecWrapper

`VecWrapper` runs a Python list of environments synchronously. Pass one action per environment and receive one result per environment.

In [ ]:
vec_env = VecWrapper([OneHotObsWrapper(make_colour_grid_env()) for _ in range(2)])
obs_list, info_list = vec_env.reset(seed=10)
print("n_envs:", vec_env.n_envs)
print("reset obs shapes:", [obs.shape for obs in obs_list])

next_obs, rewards, terminated, truncated, infos = vec_env.step([0, 1])
print("step obs shapes:", [obs.shape for obs in next_obs])
print("step rewards:", rewards)
print("step terminated:", terminated)
print("step truncated:", truncated)

assert vec_env.n_envs == 2
assert len(obs_list) == len(info_list) == 2
assert len(next_obs) == len(rewards) == len(terminated) == len(truncated) == len(infos) == 2
vec_env.close()

## NormWrapper

`NormWrapper` normalizes Box observations for a single non-vectorized environment. Here we normalize observations but leave rewards untouched so the reward column stays easy to read.

In [ ]:
norm_env = NormWrapper(
    make_cartpole_env(),
    norm_obs=True,
    norm_rew=False,
    training=True,
)

obs, info = norm_env.reset(seed=0)
print("observation_space:", norm_env.observation_space)
print("normalized obs shape:", obs.shape)
print("obs_rms mean shape:", norm_env.obs_rms.mean.shape)
print("obs_rms count:", norm_env.obs_rms.count)

next_obs, reward, terminated, truncated, info = norm_env.step(np.array([0.0], dtype=np.float32))
print("next obs shape:", next_obs.shape)
print("reward:", reward)

assert obs.shape == (4,)
assert next_obs.shape == (4,)
assert reward == 1.0
norm_env.close()

## VecNormWrapper

`VecNormWrapper` applies the same normalization idea to vectorized environments. It expects a `DummyVecWrapper` or `VecWrapper` underneath.

In [ ]:
vec_norm_env = VecNormWrapper(
    VecWrapper([make_cartpole_env(), make_cartpole_env()]),
    norm_obs=True,
    norm_rew=False,
    training=True,
)

obs_list, info_list = vec_norm_env.reset(seed=0)
print("n_envs:", vec_norm_env.n_envs)
print("reset array shape:", np.asarray(obs_list).shape)
print("obs_rms mean shape:", vec_norm_env.obs_rms.mean.shape)

actions = [np.array([0.0], dtype=np.float32), np.array([0.0], dtype=np.float32)]
next_obs, rewards, terminated, truncated, infos = vec_norm_env.step(actions)
print("step array shape:", np.asarray(next_obs).shape)
print("rewards:", rewards)

assert vec_norm_env.n_envs == 2
assert np.asarray(obs_list).shape == (2, 4)
assert np.asarray(next_obs).shape == (2, 4)
assert len(rewards) == 2
vec_norm_env.close()

## FlattenDictObsWrapper

Some wrappers produce dict observations. For example, `ltl_safety` with `obs_type="dict"` returns the original state and automaton state separately. Apply `OneHotObsWrapper` first so each discrete field becomes a Box vector, then flatten the dict into one `Box(83,)` vector.

In [ ]:
dict_env = make_colour_bomb_ltl_dict_env()
raw_obs, info = dict_env.reset(seed=0)
print("raw dict observation_space:", dict_env.observation_space)
print("raw obs:", raw_obs)
assert set(raw_obs.keys()) == {"orig", "automaton"}

dict_env = OneHotObsWrapper(dict_env)
onehot_obs, info = dict_env.reset(seed=0)
print("one-hot dict observation_space:", dict_env.observation_space)
print("one-hot dict shapes:", {key: value.shape for key, value in onehot_obs.items()})

flat_env = FlattenDictObsWrapper(dict_env)
flat_obs, info = flat_env.reset(seed=0)
print("flat observation_space:", flat_env.observation_space)
print("flat obs shape:", flat_obs.shape)
print("flat obs sum:", float(flat_obs.sum()))

next_obs, reward, terminated, truncated, info = flat_env.step(0)
print("step obs shape:", next_obs.shape)

assert flat_env.observation_space.shape == (83,)
assert flat_obs.shape == (83,)
assert next_obs.shape == (83,)
assert flat_obs.sum() == 2.0
flat_env.close()

## Wrap-up

`OneHotObsWrapper` and `FlattenDictObsWrapper` make observation spaces neural-policy friendly. `DummyVecWrapper` and `VecWrapper` make environment outputs batch-like. `NormWrapper` and `VecNormWrapper` maintain running normalization statistics for single and vectorized Box-observation environments.